In [ ]:
""" Preliminaries, for developing and debugging in a python notebook.
    None of this is strictly necessary, but makes development faster and easier.
"""

# Set up the path so our code is accessible w/o installing as a package.
import sys, os
if os.path.abspath("../src") not in sys.path:
    sys.path.insert(0, os.path.abspath("../src"))
if os.path.abspath("../src/starelib") in sys.path:
    sys.path.remove(os.path.abspath("../src/starelib"))
    
# Force automatic re-load of libraries on every call with magic
# (useful for accelerated debugging of changing library code)
%load_ext autoreload
%autoreload 2

# Verify our working directory,
# and ensure our local '.../stare_pet/src/' path is available
print(f"Executing notebook from '{os.getcwd()}'")
print("System paths available:")
for p in sys.path:
    print(f"  {p}")


In [ ]:
# Import everything for the whole notebook here rather than in each code block
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


# Define Context

Set up paths to data, file names, etc. For this notebook, we'll review the TACs for one biograph run that was well-behaved.

In [ ]:
subject = "NHFDG279"

# out_dir_name = "cerepet_resampled_2mm"
out_dir_name = "biograph_from_jjm"

# subject_dir = f"{subject}_fail_no_vasc_tacs"
subject_dir = subject

step_one_csv_file_name = f"sub-{subject}_step-1-1_kmeans_tac.csv"

subject_output_path = Path(f"/home/mike/cache/{out_dir_name}/{subject_dir}")

step_one_csv_file = subject_output_path / step_one_csv_file_name


In [ ]:
data = pd.read_csv(step_one_csv_file)
data

In [ ]:
# Just to check, what levels of K are available?
print(data['k'].unique())

In [ ]:
# And is it a string or an integer?
print(len(data[data['likely_vascular']]))
print(len(data[data['k'] == 38]))
print(len(data[((data['likely_vascular']) & (data['k'] == 38))]))


## Plot Vascular TACs by k

We probably only care about the larger values of K, and we don't have infinite real estate for plotting everything. So we'll focus only on the top 4 (k=26, 30, 34, 38) to see what they can tell us.

In [ ]:
# Create a 16:9 figure (good aspect for a computer screen) 
# with four columns (one for each value of K) 
# and three rows (one for vascular, one for irreversible, one for noise).
fig, axes = plt.subplots(
    ncols=4, nrows=3, figsize=(16, 9), layout='tight', sharey=True
)

# Loop over our characteristics and plot the 12 panels
for row, feature in enumerate(["likely_vascular", "likely_irreversible", "likely_noise", ]):
    for col, k in enumerate([26, 30, 34, 38, ]):
        ax = axes[row, col]
        subdata = data[((data[feature]) & (data['k'] == k) & (data['t'] < 20.0))]
        if len(subdata) > 0:
            sns.lineplot(
                data=subdata, x='t', y='activity', hue='label', ax=ax,
            )
        ax.set_title(f"{feature.split('_')[1].capitalize()} TACs for K={k}")


In [ ]:
fig.savefig("cluster_tacs_by_k_and_features.png")
